<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SQL_Advanced_subqueries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Advanced: Subqueries

Hello! So far, you’ve learned how to build SQL queries using a variety of clauses to filter, sort, and combine data.

Now it’s time to take the next step.

In this section, you’ll learn about **subqueries**, a powerful SQL feature that allows you to write queries **inside other queries**. With subqueries, you can solve more complex problems and build more flexible SQL instructions.

Ready to dive in? Let’s get started.


## **SQL Environment Setup (do not edit)**

In [1]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import duckdb
import pandas as pd
from pathlib import Path


In [2]:
# @title

DB_FILE = 'class_joins.duckdb'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = duckdb.connect(DB_FILE)

conn.execute(r'''
DROP TABLE IF EXISTS hiking_trip;
DROP TABLE IF EXISTS trip;
DROP TABLE IF EXISTS mountain;
DROP TABLE IF EXISTS city;
DROP TABLE IF EXISTS country;

CREATE TABLE country (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    population INTEGER NOT NULL,
    area INTEGER NOT NULL
);

CREATE TABLE city (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    country_id INTEGER NOT NULL,
    population INTEGER NOT NULL,
    area INTEGER NOT NULL,
    rating INTEGER NOT NULL,
    FOREIGN KEY (country_id) REFERENCES country(id)
);

CREATE TABLE mountain (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    height INTEGER NOT NULL,
    country_id INTEGER NOT NULL,
    FOREIGN KEY (country_id) REFERENCES country(id)
);

CREATE TABLE trip (
    id INTEGER PRIMARY KEY,
    city_id INTEGER NOT NULL,
    days INTEGER NOT NULL,
    price INTEGER NOT NULL,
    FOREIGN KEY (city_id) REFERENCES city(id)
);

CREATE TABLE hiking_trip (
    id INTEGER PRIMARY KEY,
    mountain_id INTEGER NOT NULL,
    days INTEGER NOT NULL,
    price INTEGER NOT NULL,
    length INTEGER NOT NULL,
    difficulty INTEGER NOT NULL,
    FOREIGN KEY (mountain_id) REFERENCES mountain(id)
);

INSERT INTO country VALUES
(1, 'France', 66600000, 640680),
(2, 'Germany', 80700000, 357000),
(3, 'Liechtenstein', 37340, 160),
(4, 'Spain', 46464000, 505990),
(5, 'Italy', 60795000, 301300);

INSERT INTO city VALUES
(1, 'Paris', 1, 2243000, 102, 5),
(2, 'Marseille', 1, 850700, 240, 4),
(3, 'Lyon', 1, 484300, 48, 4),
(4, 'Berlin', 2, 3460000, 888, 3),
(5, 'Hamburg', 2, 1786000, 755, 3),
(6, 'Munich', 2, 1353000, 310, 4),
(7, 'Vaduz', 3, 5400, 17, 3),
(8, 'Madrid', 4, 3165000, 605, 5),
(9, 'Barcelona', 4, 1620000, 102, 5),
(10, 'Valencia', 4, 809000, 135, 3),
(11, 'Rome', 5, 2869000, 1285, 5),
(12, 'Milan', 5, 1337000, 180, 3);

INSERT INTO mountain VALUES
(1, 'Mont Blanc', 4808, 1),
(2, 'Barre des Ecrins', 4100, 1),
(3, 'Zugspitze', 2962, 2),
(4, 'Schneefernerkopf', 2874, 2),
(5, 'Naafkopf', 2570, 3),
(6, 'Mulhacen', 3478, 4),
(7, 'Corno Grande', 2912, 5),
(8, 'Monte Amaro', 2793, 5);

INSERT INTO trip VALUES
(1, 1, 3, 1200),
(2, 1, 7, 2000),
(3, 2, 7, 1800),
(4, 2, 14, 3800),
(5, 3, 5, 1400),
(6, 4, 5, 1750),
(7, 5, 1, 300),
(8, 6, 3, 950),
(9, 6, 10, 2900),
(10, 7, 5, 1350),
(11, 8, 5, 1650),
(12, 9, 7, 1700),
(13, 10, 11, 3100),
(14, 11, 14, 3450),
(15, 12, 21, 5100);

INSERT INTO hiking_trip VALUES
(1, 1, 3, 1000, 30, 3),
(2, 1, 1, 300, 5, 1),
(3, 2, 3, 1200, 20, 2),
(4, 2, 7, 2300, 50, 4),
(5, 3, 4, 1200, 35, 4),
(6, 4, 14, 5300, 90, 5),
(7, 5, 7, 2500, 45, 4),
(8, 6, 3, 1100, 32, 3),
(9, 7, 2, 600, 21, 2),
(10, 8, 2, 600, 23, 3);
''')
print(f"Duckdb database ready ✅ ({DB_FILE})")


Duckdb database ready ✅ (class_joins.duckdb)


## Get to know the data

Complex queries require a richer dataset, so for this section we’ll work with a small **travel agency database**.

The database contains five tables:

- `country`
- `city`
- `mountain`
- `trip`
- `hiking_trip`

Use the **Database** button on the right to explore the contents of each table as we go through them.


### Table `country`

Let’s start with the table `country`.

It contains a list of countries where the travel agency operates. For each country, the table stores some basic information:

- `id` – the unique identifier of the country  
- `name` – the name of the country  
- `population` – the population of the country  
- `area` – the total area in square kilometers


### Tables `city` and `mountain`

Next, let’s take a look at the table `city`.

Each country contains multiple cities, so this table stores information about them:

- `id` – the unique identifier of the city  
- `name` – the name of the city  
- `country_id` – the country where the city is located  
- `population` – the population of the city  
- `area` – the area of the city in square kilometers  
- `rating` – a rating from **1 to 5**, where  
  - **5** means *an amazing place to visit*  
  - **1** means *not very interesting*

Now let’s check the table `mountain`.

Many countries also have mountains that tourists may want to visit. This table contains:

- `id` – the unique identifier of the mountain  
- `name` – the name of the mountain  
- `height` – the height of the mountain in meters  
- `country_id` – the country where the mountain is located


### Tables `trip` and `hiking_trip`

Finally, we have the tables that store information about the **trips themselves**.

#### `trip`

This table contains trips to cities. It stores:

- `id` – the unique identifier of the trip  
- `city_id` – the city where the trip takes place  
- `days` – the duration of the trip in days  
- `price` – the total price of the trip

#### `hiking_trip`

In addition to city trips, the agency also offers hiking trips in the mountains.

This table contains:

- `id` – the unique identifier of the hiking trip  
- `mountain_id` – the mountain where the trip takes place  
- `days` – the duration of the trip in days  
- `price` – the price of the trip  
- `length` – the length of the hiking route in kilometers  
- `difficulty` – the difficulty level from **1 to 5**, where  
  - **1** means *very easy*  
  - **5** means *extremely challenging*

---

Now that you know the structure of the database, let’s start exploring **subqueries**.


In [21]:
# @title ER diagram
%%html
<img id="er-img" style="width:50%; max-width:100%;  height:auto;"
     data-light="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/ER_advanced_subqueries.png"
     data-dark="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/ER_advanced_subqueries_black.png"
     alt="ER diagram">

<script>
  const img = document.getElementById("er-img");

  function isDarkTheme() {
    // Colab sets html[theme=dark] on the top document
    const themeAttr = document.documentElement.getAttribute("theme");
    if (themeAttr) return themeAttr === "dark";

    // fallback: OS/browser preference
    return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches;
  }

  function updateImage() {
    img.src = isDarkTheme() ? img.dataset.dark : img.dataset.light;
  }

  updateImage();

  // React to Colab theme toggles (attribute changes)
  new MutationObserver(updateImage).observe(document.documentElement, {
    attributes: true,
    attributeFilter: ["theme"]
  });

  // React to OS/browser theme changes (fallback)
  if (window.matchMedia) {
    const mq = window.matchMedia("(prefers-color-scheme: dark)");
    mq.addEventListener?.("change", updateImage);
    mq.addListener?.(updateImage); // older browsers
  }
</script>

## Subqueries with various logical operators

Excellent! Subqueries can also be combined with **different logical operators**.

Let’s look at the following example:

```sql
SELECT *
FROM mountain
WHERE height > (
  SELECT height
  FROM mountain
  WHERE name = 'Zugspitze'
);
````

Here’s what happens step by step:

1. The subquery runs first:

   ```sql
   SELECT height
   FROM mountain
   WHERE name = 'Zugspitze';
   ```

   It returns the height of the mountain **Zugspitze**.

2. The outer query then compares every mountain’s height with that value.

As a result, the query returns **all mountains that are higher than Zugspitze**.

In this example, we used the **greater-than operator (`>`)** together with a subquery, but other comparison operators such as `<`, `>=`, or `<=` can be used in the same way.


In [4]:
# @title Practice 1
base_practice_1 = make_df_validator_nospoilers(
    expected_hash='3c1121b8b6cc337c6c94d759377f3439da50f8ef7867a5ced48b04349d447fec',
    required_cols=['name'],
    expected_rows=10,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_1 = base_practice_1

make_sql_runner(
    conn,
    runner_id="practice_1",
    description_md='### Practice 1\nFind the names of all cities which have a population lower than Madrid.\n',
    validator=val_practice_1,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['city']
)


## Subqueries with functions

Nice! Subqueries can also include **aggregate functions**.

Take a look at the following example:

```sql
SELECT *
FROM hiking_trip
WHERE length < (
  SELECT AVG(length)
  FROM hiking_trip
);
````

Let’s see what happens here.

First, the subquery is executed:

```sql
SELECT AVG(length)
FROM hiking_trip;
```

This calculates the **average length of all hiking trips**.

Next, the outer query compares each trip’s length with that value.

As a result, the query returns **all hiking trips that are shorter than the average distance**.

In this example, we used the function `AVG()`, but other aggregate functions such as `MIN()`, `MAX()`, `SUM()`, or `COUNT()` can also be used inside subqueries.


In [5]:
# @title Practice 2
base_practice_2 = make_df_validator_nospoilers(
    expected_hash='5f4de9794bb510fa5b6032eaf62181a9224e45858bd7fa185bff1698387c0b36',
    required_cols=['id', 'city_id', 'days', 'price'],
    expected_rows=5,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_2 = base_practice_2

make_sql_runner(
    conn,
    runner_id="practice_2",
    description_md='### Practice 2\nFind all information about trips whose price is higher than the average.\n',
    validator=val_practice_2,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['trip']
)


## The operator `IN`

Excellent! These examples might be starting to feel a bit too easy. Let’s try something slightly more interesting.

So far, our subqueries returned **a single value** (for example `5` or `15.28`). But sometimes we want to compare a column with **multiple possible values**.

To do that, we can use the operator `IN`.

Take a look at this example:

```sql
SELECT *
FROM city
WHERE rating IN (3, 4, 5);
````

What does `IN` do?

The `IN` operator allows us to check whether a value **matches any value from a list**. It’s a convenient alternative to writing multiple `OR` conditions.

In our example, we want to display cities that are at least somewhat interesting. We’re not too picky — any city with a rating **3, 4, or 5** is acceptable.

So the condition:

```sql
rating IN (3, 4, 5)
```

is equivalent to writing:

```sql
rating = 3 OR rating = 4 OR rating = 5
```

Using `IN` simply makes the query **shorter and easier to read**.


In [6]:
# @title Practice 3
base_practice_3 = make_df_validator_nospoilers(
    expected_hash='bf9ee45c296a8adc8a60b274f589ddc7b65c71fbf1d4cacb50b42fc3cc683d0c',
    required_cols=['id', 'mountain_id', 'days', 'price', 'length', 'difficulty'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_3 = base_practice_3

make_sql_runner(
    conn,
    runner_id="practice_3",
    description_md='### Practice 3\nFind all information about hiking trips with difficulty 1, 2, or 3.\n',
    validator=val_practice_3,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['hiking_trip']
)


## The operator `IN` with subqueries

Great! Now let’s combine the `IN` operator with **subqueries**.

Consider the following example:

```sql
SELECT price
FROM trip
WHERE city_id IN (
  SELECT id
  FROM city
  WHERE population < 2000000
);
````

Let’s break it down.

First, the subquery runs:

```sql
SELECT id
FROM city
WHERE population < 2000000;
```

This returns the **IDs of all cities with a population below 2 million**.

Next, the outer query uses those IDs inside the `IN` operator.

As a result, the query returns the **prices of trips to cities whose population is less than 2 million**.

In other words, the condition:

```sql
city_id IN (...)
```

checks whether the `city_id` of a trip matches **any city returned by the subquery**.


In [7]:
# @title Practice 4
base_practice_4 = make_df_validator_nospoilers(
    expected_hash='753b431802e434d02149c390c849ac0a9b2c71ab60712ee6a137d6658a5ca52f',
    required_cols=['id', 'city_id', 'days', 'price'],
    expected_rows=13,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_4 = base_practice_4

make_sql_runner(
    conn,
    runner_id="practice_4",
    description_md='### Practice 4\nFind all information about all trips in cities whose area is greater than 100.\n',
    validator=val_practice_4,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['trip', 'city']
)


## The operator `ALL`

Outstanding! Since you're doing so well, let’s introduce another useful operator: `ALL`.

Consider the following example:

```sql
SELECT *
FROM country
WHERE area > ALL (
  SELECT area
  FROM city
);
````

Here we use the operator `ALL` together with the comparison operator `>`.

The expression:

```sql
area > ALL (...)
```

means that the value of `area` must be **greater than every value returned by the subquery**.

Let’s see what happens step by step:

1. The subquery runs first:

```sql
SELECT area
FROM city;
```

This returns the **areas of all cities**.

2. The outer query then compares each country’s area with **all those values**.

As a result, the query returns **countries whose area is larger than the area of every city**.

For example, Liechtenstein is a very small country. Although it is larger than some cities (such as Lyon), it is not larger than every city (Berlin, for instance, is larger). Therefore, Liechtenstein would **not** appear in the result.

---

The operator `ALL` can also be used with other comparison operators, such as:

* `= ALL`
* `!= ALL` or `<> ALL`
* `< ALL`
* `<= ALL`
* `>= ALL`


In [8]:
# @title Practice 5
base_practice_5 = make_df_validator_nospoilers(
    expected_hash='43aa987663f27b3512bc534a2bd379723f9b2d7546d238a0b7fc301b79da0264',
    required_cols=['id', 'name', 'country_id', 'population', 'area', 'rating'],
    expected_rows=1,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_5 = base_practice_5

make_sql_runner(
    conn,
    runner_id="practice_5",
    description_md='### Practice 5\nFind all information about the cities which are less populated than all countries in the database\n',
    validator=val_practice_5,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['city', 'country']
)


## The operator `ANY`

Good. To conclude this section, let’s introduce one more operator: `ANY`.

Take a look at the following example:

```sql
SELECT *
FROM trip
WHERE price < ANY (
  SELECT price
  FROM hiking_trip
  WHERE mountain_id = 1
);
````

Let’s break it down.

First, the subquery runs:

```sql
SELECT price
FROM hiking_trip
WHERE mountain_id = 1;
```

This returns the **prices of all hiking trips to mountain `1`** (Mont Blanc).
In our dataset, there are two such trips: one costs **1000** and the other **300**.

Next, the outer query checks the condition:

```sql
price < ANY (...)
```

The operator `ANY` means that the condition must be true for **at least one value returned by the subquery**.

So a city trip will appear in the result if its price is **lower than at least one of the hiking trip prices**.

For example:

* if a city trip costs **250**, it is cheaper than **300**, so it will appear in the result
* if a city trip costs **900**, it is cheaper than **1000**, so it will also appear
* if a city trip costs **1200**, it is not cheaper than any of them, so it will not appear

---

Just like with `ALL`, the operator `ANY` can be used with other comparison operators:

* `= ANY`
* `!= ANY` or `<> ANY`
* `< ANY`
* `<= ANY`
* `> ANY`
* `>= ANY`


In [9]:
# @title Practice 6
base_practice_6 = make_df_validator_nospoilers(
    expected_hash='3bfac7f33b8ccbb935b65fd28ada1230b558f087f110e21dad6094504e702ea1',
    required_cols=['id', 'city_id', 'days', 'price'],
    expected_rows=2,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_6 = base_practice_6

make_sql_runner(
    conn,
    runner_id="practice_6",
    description_md='### Practice 6\nFind all information about all the city trips which have the same price as any hiking trip.\n',
    validator=val_practice_6,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['trip', 'hiking_trip']
)


## Get to know correlated subqueries

Very good. So far, the subqueries we used were **independent** of the main query.  
That means you could run the subquery on its own, take its result, and then use it in the main query.

Now we’ll look at a different kind of subquery: a **correlated subquery**.

A correlated subquery **depends on the outer query** and cannot run independently.

Study the following example:

```sql
SELECT *
FROM country
WHERE area <= (
  SELECT MIN(area)
  FROM city
  WHERE city.country_id = country.id
);
````

Let’s see what this query does.

For each country, the subquery finds the **smallest city area within that country**.
The outer query then checks whether the **country’s area is smaller than or equal to that value**.

In other words, the query looks for countries that are **smaller than their smallest city**.

Why would we write such a query?
It can be useful for **data validation**. If the query returns any rows, it likely means there is an error in the data — a country cannot realistically be smaller than one of its cities.

---

### What makes this a correlated subquery?

Look closely at the condition inside the subquery:

```sql
WHERE city.country_id = country.id
```

Here, the subquery refers to `country.id`, which comes from the **outer query**.

If you tried to run the subquery alone, the database would not know which `country.id` to use, because the table `country` is not part of the subquery.

But when the query runs, SQL processes it like this:

1. The outer query selects a row from `country`.
2. The subquery runs using that country’s `id`.
3. The condition is evaluated.
4. The process repeats for the next country.

---

### Golden rule

* A **subquery can reference tables from the outer query**.
* But the **outer query cannot reference tables from the subquery**.


In [10]:
# @title Practice 7
base_practice_7 = make_df_validator_nospoilers(
    expected_hash='5713493e675b659b1c44fb579dd443f84841956a1d2ba1dd1a1bd6b499dce853',
    required_cols=['id', 'name', 'population', 'area'],
    expected_rows=0,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_7 = base_practice_7

make_sql_runner(
    conn,
    runner_id="practice_7",
    description_md="### Practice 7\nLet's check if the database contains any errors in a sample exercise.\n\nFind all information about each country whose population is equal to or smaller than the population of the least populated city in that specific country.\n",
    validator=val_practice_7,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['country', 'city']
)


## Aliases for tables

Great work! Correlated subqueries can be tricky, so well done for making it this far.

Sometimes the **same table appears both in the outer query and in the correlated subquery**. In such cases, we need a way to distinguish between the two uses of the table.

Consider the following example:

```sql
SELECT *
FROM city main_city
WHERE population > (
  SELECT AVG(population)
  FROM city average_city
  WHERE average_city.country_id = main_city.country_id
);
````

Let’s understand what this query does.

We want to find **cities whose population is greater than the average population of cities in the same country**.

To do this:

1. The outer query selects a city (`main_city`).
2. The subquery calculates the **average population of all cities in that same country**.
3. The outer query checks whether the selected city’s population is **greater than that average**.

---

### Why do we need aliases?

Notice that the table `city` appears **twice** in this query:

* once in the main query
* once in the subquery

If we didn’t use aliases, SQL would not know which instance of the table we are referring to.

To solve this, we give each instance a **temporary name (alias)**:

* `city main_city` → the table used in the outer query
* `city average_city` → the table used in the subquery

These aliases allow us to clearly distinguish between the two.

The alias is written **after the table name**, separated by a space:

```sql
FROM city main_city
```

No commas are used, just the table name followed by its alias.


In [11]:
# @title Practice 8
base_practice_8 = make_df_validator_nospoilers(
    expected_hash='f1cc29f51ec1e6b3e12673f41d5547b28297cfcf06d650c6967c97cb1f8ecd3b',
    required_cols=['id', 'name', 'country_id', 'population', 'area', 'rating'],
    expected_rows=5,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_8 = base_practice_8

make_sql_runner(
    conn,
    runner_id="practice_8",
    description_md='### Practice 8\nFind all information about cities with a rating higher than the average rating for all cities in that specific country.\n',
    validator=val_practice_8,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['city']
)


## The operator `IN` with correlated subqueries

Great! Earlier we learned about the operator `IN`, which allows us to check whether a value matches **any value from a list or a subquery result**.

Now let’s see how `IN` can also be used with **correlated subqueries**.

Take a look at the following example:

```sql
SELECT *
FROM city
WHERE country_id IN (
  SELECT id
  FROM country
  WHERE country.population - 40000 < city.population
);
````

Let’s analyze what happens here.

The subquery searches for **countries whose population is not much larger than the population of a given city**. Notice that the subquery refers to `city.population`, which comes from the outer query — this makes it a **correlated subquery**.

For each city, SQL checks whether its `country_id` appears among the countries returned by the subquery.

As a result, the query returns **cities whose population is within 40,000 people of their country’s population**.

In other words, it lists cities from countries where the country's population exceeds the city's population by **40,000 or less**.


In [12]:
# @title Practice 9
base_practice_9 = make_df_validator_nospoilers(
    expected_hash='e9ca28a3792e063b9cd236d626bc731068cfca1a0d80ec59169a43464f2976de',
    required_cols=['id', 'city_id', 'days', 'price'],
    expected_rows=1,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_9 = base_practice_9

make_sql_runner(
    conn,
    runner_id="practice_9",
    description_md='### Practice 9\nShow all information about all trips to cities where the ratio of city area to trip duration (in days) is greater than 700.\n',
    validator=val_practice_9,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['trip', 'city']
)


## The operator `EXISTS`

Awesome! Let’s learn another useful operator: `EXISTS`.

Take a look at the following example:

```sql
SELECT *
FROM city
WHERE EXISTS (
  SELECT *
  FROM trip
  WHERE city_id = city.id
);
````

The operator `EXISTS` checks whether a subquery returns **at least one row**.

Let’s see how this query works:

1. The outer query selects a city.
2. The subquery checks whether there is **any trip associated with that city**.
3. If the subquery finds at least one matching row, `EXISTS` evaluates to **true**, and the city appears in the result.

As a result, this query returns **only cities for which the travel agency organizes at least one trip**.

Cities that have **no trips** in the `trip` table will not appear in the result.


In [13]:
# @title Practice 10
base_practice_10 = make_df_validator_nospoilers(
    expected_hash='93fe1b0af56036cb0b67460cb9a8780e1fcc7e29c9b010ec7a39638a28858da4',
    required_cols=['id', 'name', 'population', 'area'],
    expected_rows=5,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_10 = base_practice_10

make_sql_runner(
    conn,
    runner_id="practice_10",
    description_md='### Practice 10\nSelect all countries where there is at least one mountain.\n',
    validator=val_practice_10,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['country', 'mountain']
)


## The operator `EXISTS` with `NOT`

Good! Remember the operator `NOT`? We can combine it with `EXISTS` as well.

Take a look at the following example:

```sql
SELECT *
FROM city
WHERE NOT EXISTS (
  SELECT *
  FROM trip
  WHERE city_id = city.id
);
````

As you might expect, `NOT EXISTS` does the opposite of `EXISTS`.

Instead of checking whether the subquery returns **at least one row**, it checks that the subquery returns **no rows at all**.

Here’s how the query works:

1. The outer query selects a city.
2. The subquery checks whether there are any trips associated with that city.
3. If **no trips are found**, the condition `NOT EXISTS` becomes true.

As a result, the query returns **all cities that do not have any trips organized to them**.


In [14]:
# @title Practice 11
base_practice_11 = make_df_validator_nospoilers(
    expected_hash='c6acb4bc7ae25c81cc51bbbaaf4c65bedd452771a64aafef12dd98bb73708da3',
    required_cols=['id', 'name', 'height', 'country_id'],
    expected_rows=0,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_11 = base_practice_11

make_sql_runner(
    conn,
    runner_id="practice_11",
    description_md='### Practice 11\nSelect all mountains with no hiking trips to them.\n',
    validator=val_practice_11,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['mountain', 'hiking_trip']
)


## The operator `ALL` in correlated subqueries

Good. Remember the operator `ALL`? Let’s now use it inside a **correlated subquery**.

Consider the following example:

```sql
SELECT *
FROM trip main_trip
WHERE price >= ALL (
  SELECT price
  FROM trip sub_trip
  WHERE main_trip.city_id = sub_trip.city_id
);
````

Let’s see how this query works.

For each trip in the outer query (`main_trip`), the subquery finds **all trip prices for the same city**.

The condition:

```sql
price >= ALL (...)
```

means that the trip’s price must be **greater than or equal to every other trip price in that city**.

As a result, the query returns **the most expensive trips for each city**. If several trips share the same highest price, all of them will appear in the result.


In [15]:
# @title Practice 12
base_practice_12 = make_df_validator_nospoilers(
    expected_hash='2e55e5959f7c1936778aac19002e5ca6af0d4ef83fc8a6e47aa8e81a5b7192d8',
    required_cols=['id', 'mountain_id', 'days', 'price', 'length', 'difficulty'],
    expected_rows=8,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_12 = base_practice_12

make_sql_runner(
    conn,
    runner_id="practice_12",
    description_md='### Practice 12\nSelect the hiking trip with the longest distance (column length) for every mountain\n',
    validator=val_practice_12,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['hiking_trip']
)


## Subqueries in the `FROM` clause

Good job! Subqueries can also appear in places other than the `WHERE` clause. For example, we can use a subquery **as a temporary table** in the `FROM` clause.

Take a look at the following example:

```sql
SELECT *
FROM city, (
    SELECT *
    FROM country
    WHERE area < 1000
) AS small_country
WHERE small_country.id = city.country_id;
````

In this query, the subquery creates a **temporary result set** containing only countries with an area smaller than 1,000 square kilometers.

Because this result set does not exist as a real table in the database, we must give it a name. We do this using an **alias**, in this case:

```sql
AS small_country
```

The outer query then treats `small_country` like a normal table.

Finally, the condition:

```sql
small_country.id = city.country_id
```

connects the two tables, so that each city is matched with its country.

As a result, the query returns **cities located in countries with an area smaller than 1,000 square kilometers**.

Remember: when selecting from multiple tables, we need a condition that links them. Without it, SQL would combine **every city with every country**, producing a Cartesian product.


In [16]:
# @title Practice 13
base_practice_13 = make_df_validator_nospoilers(
    expected_hash='f47b01b704cd53b120ecde3a5fefb748acfad9bc6a39310a71aa5417aa0f37d3',
    required_cols=['id', 'name', 'height', 'country_id', 'id_1', 'name_1', 'population', 'area'],
    expected_rows=7,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_13 = base_practice_13

make_sql_runner(
    conn,
    runner_id="practice_13",
    description_md='### Practice 13\nShow mountains together with their countries. The countries must have at least 50,000 residents.\n',
    validator=val_practice_13,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['mountain', 'country']
)


## Subqueries in the `FROM` clause

Great! Of course, when using subqueries in the `FROM` clause, you don’t have to select every column. You can still choose **only the columns you need** in the outer query.

Consider the following example:

```sql
SELECT
  name,
  days,
  price
FROM trip, (
  SELECT *
  FROM city
  WHERE rating = 5
) AS nice_city
WHERE nice_city.id = trip.city_id;
````

In this query, the subquery creates a temporary table called `nice_city`. It contains **only cities with a rating of 5**.

The outer query then joins this temporary table with the `trip` table and shows information about trips to those highly rated cities.

As a result, the query returns:

* the **city name**
* the **duration of the trip** (`days`)
* the **trip price**

for trips that take place in cities with a rating of 5.

---

### Note

When the selected tables have **no columns with the same name**, you can omit the table name prefix.

For example:

```sql
price
```

instead of:

```sql
trip.price
```

This works because there is only **one column named `price`** in the query result, so SQL can determine which column you mean.


In [17]:
# @title Practice 14
base_practice_14 = make_df_validator_nospoilers(
    expected_hash='484dd71f310c3f1a71918a245fe9c9403aa6fa128d68b18783e83156bd8ed81b',
    required_cols=['length', 'height'],
    expected_rows=5,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_14 = base_practice_14

make_sql_runner(
    conn,
    runner_id="practice_14",
    description_md='### Practice 14\nShow hiking trips together with their mountains. The mountains must be at least 3,000 high. Select only the columns length and height.\n',
    validator=val_practice_14,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['hiking_trip', 'mountain']
)


## Subqueries in the `SELECT` clause

Awesome! Subqueries can also appear in the **column list of a `SELECT` clause**.

When used in this position, the subquery must return **exactly one value** (one row and one column) for each row processed by the main query.

Take a look at the following example:

```sql
SELECT
  name,
  (
    SELECT COUNT(*)
    FROM trip
    WHERE city_id = city.id
  ) AS trip_count
FROM city;
````

Let’s see what happens here.

The outer query goes through each row of the `city` table and selects the **city name**.

For every city, the subquery runs and counts how many trips in the `trip` table have a matching `city_id`.

As a result, the query returns:

* the **name of each city**
* the **number of trips organized to that city**

The result might look like this:

| name   | trip_count |
| ------ | ---------- |
| Paris  | 3          |
| Rome   | 2          |
| Berlin | 0          |

Here, the function `COUNT()` is used to calculate the number of trips for each city.


In [18]:
# @title Practice 15
base_practice_15 = make_df_validator_nospoilers(
    expected_hash='b88e91160cdd92c836860107a726dd4ccb5fa1f0e269e168abe896e7184a30a3',
    required_cols=['name', 'count'],
    expected_rows=8,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_15 = base_practice_15

make_sql_runner(
    conn,
    runner_id="practice_15",
    description_md='### Practice 15\nShow each mountain name together with the number of hiking trips to that mountain (name the column count).\n',
    validator=val_practice_15,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['mountain', 'hiking_trip']
)
